In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# ── Carga ─────────────────────────────────────────────────────────────────────
orders    = pd.read_csv('orders_dataset.csv')
customers = pd.read_csv('customers_dataset.csv')
payments  = pd.read_csv('order_payments_dataset.csv')

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
fecha_ref = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# ── RFM ───────────────────────────────────────────────────────────────────────
df    = orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')
pagos = payments.groupby('order_id')['payment_value'].sum().reset_index()
df    = df.merge(pagos, on='order_id', how='left')

rfm = df.groupby('customer_unique_id').agg(
    recency   = ('order_purchase_timestamp', lambda x: (fecha_ref - x.max()).days),
    frequency = ('order_id', 'count'),
    monetary  = ('payment_value', 'sum')
).reset_index()

# ── Clustering ────────────────────────────────────────────────────────────────
rfm_scaled     = StandardScaler().fit_transform(rfm[['recency', 'frequency', 'monetary']])
rfm['cluster'] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(rfm_scaled)

rfm['segmento'] = rfm['cluster'].map({
    0: 'Inactivo',
    1: 'Comprador reciente',
    2: 'Alto ticket',
    3: 'Comprador recurrente'
})

# ── Tabla resumen ─────────────────────────────────────────────────────────────
tabla = rfm.groupby('segmento').agg(
    clientes  = ('customer_unique_id', 'count'),
    recency   = ('recency', 'mean'),
    frequency = ('frequency', 'mean'),
    monetary  = ('monetary', 'mean')
).round(1)
tabla['% base'] = (tabla['clientes'] / tabla['clientes'].sum() * 100).round(1)

print("SEGMENTACIÓN RFM — PERFIL POR SEGMENTO:")
print(tabla.to_string())


SEGMENTACIÓN RFM — PERFIL POR SEGMENTO:
                      clientes  recency  frequency  monetary  % base
segmento                                                            
Alto ticket               2422    289.5        1.0    1196.2     2.5
Comprador reciente       52056    178.4        1.0     135.2    54.2
Comprador recurrente      2962    269.3        2.1     290.3     3.1
Inactivo                 38656    438.8        1.0     134.9    40.2
